# FT-Code training on Colab T4

Trains FT-Code-300, FT-Code-1000, FT-Code-5000 from CodeSearchNet/python on a T4 runtime. Mounts Drive at `/content/drive/MyDrive/adaptmem-bench/ft-code/` to persist checkpoints.

Expected runtime on T4: ~5 min per FT-Code-300, ~15 min per FT-Code-1000, ~75 min per FT-Code-5000. Total ~95 min including data download + indexing.

Verify GPU at the top, run end-to-end. Each train cell saves its own checkpoint, so a runtime interruption only loses the cell in flight.

In [ ]:
# 1. Sanity: confirm T4 GPU is attached
import torch
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Switch runtime to GPU (Runtime > Change runtime type > T4 GPU)'

In [ ]:
# 2. Install dependencies and clone the repo (public)
!pip install -q sentence-transformers datasets numpy
!git clone --depth 1 https://github.com/nakata-app/adaptmem.git /content/adaptmem
%cd /content/adaptmem
!pip install -q -e .

In [ ]:
# 3. Mount Drive for checkpoint persistence
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/adaptmem-bench/ft-code'
os.makedirs(OUT_DIR, exist_ok=True)
print('checkpoints will land in', OUT_DIR)

In [ ]:
# 4. Run the training script for all three sizes
import os, sys, json, time
from pathlib import Path
sys.path.insert(0, '/content/adaptmem/benchmarks')
from codesearchnet_train_colab import train_one

os.environ['ADAPTMEM_DEVICE'] = 'cuda'
out = Path(OUT_DIR)
results = []
for n in [300, 1000, 5000]:
    print(f'\n--- starting FT-Code-{n} ---')
    t0 = time.time()
    r = train_one(n, out, device='cuda')
    print(f'FT-Code-{n} done in {time.time()-t0:.1f}s')
    results.append(r)

summary_path = out / 'training_summary.json'
summary_path.write_text(json.dumps(results, indent=2, default=str))
print(f'\nALL DONE → {summary_path}')

In [ ]:
# 5. Optional: quick eval on a 1000-query test sample for each checkpoint
from codesearchnet_eval import evaluate
from pathlib import Path

for n in [300, 1000, 5000]:
    ckpt = Path(OUT_DIR) / f'ft-code-{n}'
    res_path = Path(OUT_DIR) / f'eval_ft-code-{n}_n1000.jsonl'
    print(f'\n--- evaluating FT-Code-{n} on 1000 test queries ---')
    evaluate(ckpt, n=1000, out=res_path)

In [ ]:
# 6. Optional: full 19k-query eval for the final report. Slower; ~3-5 min per checkpoint on T4.
for n in [300, 1000, 5000]:
    ckpt = Path(OUT_DIR) / f'ft-code-{n}'
    res_path = Path(OUT_DIR) / f'eval_ft-code-{n}_full.jsonl'
    print(f'\n--- evaluating FT-Code-{n} on full test split ---')
    evaluate(ckpt, n=-1, out=res_path)